# 09 · Maximum flow and minimum cut

**Goal:** implement an augmenting-path algorithm, write max flow as an LP, and verify **max-flow = min-cut** through LP duality.

$$\max\ f\quad\text{s.t.}\quad Nx = f(e_s-e_t),\quad 0\le x\le u .$$

A trick that keeps the LP tidy: add a return arc $t\to s$ with unlimited capacity and maximize its flow, so that every node has flow conservation.

**Dual:** $\min\sum u_{ij}y_{ij}$ s.t. $y_{ij}\ge p_j-p_i$, $p_t-p_s=1$, $y\ge0$. An integral optimum has $p\in\{0,1\}$, and $y_{ij}=1$ exactly on the arcs crossing the cut, so the dual is a minimum cut.

In [ ]:
import numpy as np
import networkx as nx
from collections import deque
from scipy.optimize import linprog

CAPS = [(0, 1, 10), (0, 2, 8), (1, 2, 3), (1, 3, 6), (2, 4, 9), (1, 4, 4),
        (3, 5, 7), (4, 3, 5), (4, 6, 6), (3, 6, 2), (5, 7, 8), (6, 7, 9)]
n, s, t = 8, 0, 7
m = len(CAPS)
u = np.array([a[2] for a in CAPS], float)

## 1. Edmonds–Karp (shortest augmenting paths)

Repeatedly find an $s$–$t$ path in the **residual graph** with BFS and push the bottleneck capacity. Using shortest paths bounds the work at $O(VE^2)$.

In [ ]:
def edmonds_karp(n, caps, s, t):
    Cr = np.zeros((n, n))                      # residual capacities
    for i, j, cap in caps:
        Cr[i, j] += cap
    flow = 0.0
    while True:
        parent = [-1] * n; parent[s] = s
        q = deque([s])
        while q and parent[t] == -1:
            v = q.popleft()
            for w in np.nonzero(Cr[v] > 1e-12)[0]:
                if parent[w] == -1:
                    parent[w] = v; q.append(w)
        if parent[t] == -1:                    # no augmenting path: done
            reachable = {v for v in range(n) if parent[v] != -1}
            return flow, reachable
        path, v = [], t
        while v != s:
            path.append((parent[v], v)); v = parent[v]
        aug = min(Cr[i, j] for i, j in path)
        for i, j in path:
            Cr[i, j] -= aug; Cr[j, i] += aug   # reverse residual arc lets us undo flow
        flow += aug

f_ek, S_side = edmonds_karp(n, CAPS, s, t)
print("max flow (Edmonds-Karp):", f_ek, "   source side of min cut:", sorted(S_side))
cut_arcs = [(i, j) for i, j, _ in CAPS if i in S_side and j not in S_side]
cut_cap = sum(cap for i, j, cap in CAPS if (i, j) in cut_arcs)
print("cut arcs:", cut_arcs, "  capacity:", cut_cap)
assert np.isclose(f_ek, cut_cap)

## 2. The LP, checked against networkx

In [ ]:
Ninc = np.zeros((n, m + 1))
for k, (i, j, _) in enumerate(CAPS):
    Ninc[i, k] += 1; Ninc[j, k] -= 1
Ninc[t, m] += 1; Ninc[s, m] -= 1               # return arc t -> s
obj = np.zeros(m + 1); obj[m] = -1.0            # maximize flow on the return arc
bounds = [(0, cap) for cap in u] + [(0, None)]
lp = linprog(obj, A_eq=Ninc, b_eq=np.zeros(n), bounds=bounds, method="highs")
f_lp = -lp.fun

G = nx.DiGraph()
for i, j, cap in CAPS:
    G.add_edge(i, j, capacity=cap)
f_nx = nx.maximum_flow_value(G, s, t)
print("LP:", f_lp, "  networkx:", f_nx, "  Edmonds-Karp:", f_ek)
assert np.isclose(f_lp, f_nx) and np.isclose(f_lp, f_ek)

## 3. The dual LP is a minimum cut

In [ ]:
# variables: p (n node labels), y (m arc indicators)
A_ub = np.zeros((m, n + m))
for k, (i, j, _) in enumerate(CAPS):
    A_ub[k, j] = 1.0; A_ub[k, i] = -1.0; A_ub[k, n + k] = -1.0     # p_j - p_i - y_ij <= 0
obj = np.r_[np.zeros(n), u]
bnds = [(0, 0) if v == s else (1, 1) if v == t else (None, None) for v in range(n)] + [(0, None)] * m
dual = linprog(obj, A_ub=A_ub, b_ub=np.zeros(m), bounds=bnds, method="highs")
print("min cut via dual LP:", dual.fun)
assert np.isclose(dual.fun, f_lp)              # strong duality: max flow = min cut
y = dual.x[n:]
print("arcs with y = 1:", [CAPS[k][:2] for k in range(m) if y[k] > 0.5])

## Exercises

1. Replace BFS by DFS (Ford–Fulkerson). Build a 4-node graph with capacities $10^6$ and one capacity-1 arc where DFS needs $2\cdot10^6$ augmentations.
2. Node capacities: split each node $v$ into $v_{in}\to v_{out}$ with capacity $u_v$. Implement and test.
3. This repo's `max_flow_interdiction/` folder attacks this problem from the interdictor's side. After notebook 12, read it and write down its MILP.